# semantic_search/04 — Predict clinical characteristics from embeddings

Trains elastic-net logistic regression and XGBoost classifiers from the single concatenated 3×768 embedding representation.

**Runs after** `semantic_search/01_aggregate`. It does not depend on clustering. The default uses unrestricted **all-time** embeddings for cancer type, stage, first-treatment category, and the LLM-derived conventional/AVPC/NEPC prostate phenotype.

This is retrospective clinical-label recovery, not prospective prediction. Each setup uses nested stratified CV for out-of-fold evaluation, followed by a final fit on the complete setup cohort. Compatible artifacts are reused unless `OVERWRITE = True`.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import subprocess
import sys
import time
from pathlib import Path

from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "semantic_search").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402
from semantic_search import common  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<30} {path}")
    message = "All inputs present." if not missing else f"{len(missing)} missing: {', '.join(missing)}"
    print(f"\n{message}")
    return missing


def run_module(module: str, args: list[str] | None = None) -> int:
    """Run a module as a subprocess and stream its output."""
    cmd = [sys.executable, "-m", module] + (args or [])
    print("$ " + " ".join(cmd) + "\n", flush=True)
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=REPO_ROOT)
    print(f"\nexit={proc.returncode}  elapsed={(time.time() - t0) / 60:,.1f} min", flush=True)
    return proc.returncode


print(f"repo root:  {REPO_ROOT}")
print(f"data root:  {config.DATA_PATH}")
print(f"this arm:   {config.SEMANTIC_SEARCH_PATH}")

## Configuration

The treatment target defaults to the curated category. Set `TREATMENT_GRANULARITY = "drug"` for exact drug names. Categories with fewer than `MIN_TREATMENT_CLASS_N` matched patients are grouped into `OTHER`.

In [ ]:
MODULE = "semantic_search.train_prediction_models"

TARGETS = ["cancer_type", "stage", "first_treatment", "prostate_subtype"]
SPACES = common.SPACES
WINDOWS = common.DEFAULT_WINDOWS
MODELS = ["elastic_net", "xgboost"]

COHORT_MODE = "common"
TREATMENT_GRANULARITY = "category"
MIN_TREATMENT_CLASS_N = 25
OUTER_FOLDS = 5
INNER_FOLDS = 3
SEED = 1234
N_JOBS = min(4, os.cpu_count() or 1)
AVPC_NEPC_LABELS = config.AVPC_NEPC_LABELS_PATH
OVERWRITE = False
RUN_MODELS = True

print(f"targets:     {TARGETS}")
print(f"spaces:      {SPACES}")
print(f"windows:     {WINDOWS}")
print(f"models:      {MODELS}")
print(f"cohort mode: {COHORT_MODE}")
print(f"CV:          {OUTER_FOLDS} outer x {INNER_FOLDS} inner; n_jobs={N_JOBS}")

## Preconditions

Feature files come from `01_aggregate`. Label sources are independent: a missing source skips only that target. The prostate path is the final cohort-complete artifact from the neighboring `LLM_clinical_annotations` pipeline.

In [ ]:
feature_inputs = [
    (f"features {space}/{window}", common.feature_path(space, window))
    for window in WINDOWS for space in SPACES
]
label_inputs = [
    ("cancer-type labels", os.path.join(config.FEATURE_PATH, "cancer_type_df.csv.gz")),
    ("stage labels", os.path.join(config.FEATURE_PATH, "cancer_stage_df.csv.gz")),
    ("first-treatment cohort", os.path.join(config.SURV_PATH, "cohort_df.parquet")),
    ("LLM AVPC/NEPC labels", AVPC_NEPC_LABELS),
]
missing = check_inputs(feature_inputs + label_inputs)

print("\nPython packages:")
for package, module in [
    ("numpy", "numpy"), ("scikit-learn", "sklearn"),
    ("xgboost", "xgboost"), ("joblib", "joblib"),
]:
    ok = importlib.util.find_spec(module) is not None
    print(f"[{'ok ' if ok else 'MISSING'}] {package}")

if any(label.startswith("features ") for label in missing):
    print("\nRun 01_aggregate.ipynb for the missing feature spaces.")
if "LLM AVPC/NEPC labels" in missing:
    print("The other targets can still run; prostate_subtype will be skipped.")

## Existing model census

A complete setup has an out-of-fold prediction parquet, final model, and metadata JSON. A changed cohort, source feature file, CV setting, or software version is not silently reused.

In [ ]:
def artifact_paths(target: str, space: str, window: str, model: str) -> list[str]:
    stem = f"{target}__{space}__{window}__{model}"
    return [
        os.path.join(common.PREDICTIONS_DIR, f"{stem}.parquet"),
        os.path.join(common.MODELS_DIR, f"{stem}.joblib"),
        os.path.join(common.PREDICTION_META_DIR, f"{stem}.json"),
    ]


setups = [
    (target, space, window, model)
    for target in TARGETS for space in SPACES
    for window in WINDOWS for model in MODELS
]
complete = [
    setup for setup in setups
    if all(os.path.exists(path) for path in artifact_paths(*setup))
]
print(f"{len(complete)} / {len(setups)} requested model setups complete")
for target in TARGETS:
    n_done = sum(setup[0] == target for setup in complete)
    n_total = sum(setup[0] == target for setup in setups)
    print(f"  {target:18s} {n_done:>2d} / {n_total}")

## Run models

The default is 4 targets × 1 concatenated representation × 2 model families, with nested CV and a final refit. Completed compatible setups are reused. Narrow `TARGETS` or `MODELS` above for a smaller run.

In [ ]:
args = [
    "--targets", *TARGETS,
    "--spaces", *SPACES,
    "--windows", *WINDOWS,
    "--models", *MODELS,
    "--cohort-mode", COHORT_MODE,
    "--treatment-granularity", TREATMENT_GRANULARITY,
    "--min-treatment-class-n", str(MIN_TREATMENT_CLASS_N),
    "--outer-folds", str(OUTER_FOLDS),
    "--inner-folds", str(INNER_FOLDS),
    "--seed", str(SEED),
    "--n-jobs", str(N_JOBS),
    "--avpc-nepc-labels", str(AVPC_NEPC_LABELS),
]
if OVERWRITE:
    args.append("--overwrite")

if RUN_MODELS:
    rc = run_module(MODULE, args)
    if rc != 0:
        print("\nTraining failed — see the traceback above. Completed setups remain reusable.")
else:
    rc = None
    print("RUN_MODELS = False; skipped training and continuing to existing results.")

## Performance summary

Pooled out-of-fold metrics use every patient's held-out prediction exactly once. Fold means and standard deviations are also retained in the CSV.

In [ ]:
import polars as pl

summary_path = common.result_path("prediction_metrics_summary")
if os.path.exists(summary_path):
    summary = (
        pl.read_csv(summary_path)
        .filter(
            pl.col("target").is_in(TARGETS)
            & pl.col("space").is_in(SPACES)
            & pl.col("window").is_in(WINDOWS)
            & pl.col("model").is_in(MODELS)
        )
        .sort(
            ["target", "model", "macro_f1_pooled_oof"],
            descending=[False, False, True],
        )
    )
    display(summary.select(
        "target", "space", "window", "model", "n_patients", "n_classes",
        "balanced_accuracy_pooled_oof", "macro_f1_pooled_oof",
        "macro_ovr_auc_pooled_oof", "macro_average_precision_pooled_oof",
        "log_loss_pooled_oof",
    ))
else:
    summary = pl.DataFrame()
    print(f"No prediction summary at {summary_path}")

## Compare model families

Macro-F1 weights each class equally and is the primary compact view for imbalanced outcomes. The results table retains all other metrics.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if summary.height:
    metric = "macro_f1_pooled_oof"
    fig, axes = plt.subplots(
        1, len(TARGETS), figsize=(5 * len(TARGETS), 4), sharey=True
    )
    axes = np.atleast_1d(axes)
    x = np.arange(len(SPACES))
    width = 0.36

    for ax, target in zip(axes, TARGETS):
        target_df = summary.filter(pl.col("target") == target)
        for offset, model in zip((-width / 2, width / 2), MODELS):
            model_df = target_df.filter(pl.col("model") == model)
            lookup = dict(zip(model_df["space"].to_list(), model_df[metric].to_list()))
            values = [lookup.get(space, np.nan) for space in SPACES]
            ax.bar(x + offset, values, width, label=model.replace("_", " "))
        ax.set_title(target.replace("_", " "))
        ax.set_xticks(x, SPACES, rotation=45, ha="right")
        ax.set_ylim(0, 1)
        ax.grid(axis="y", alpha=0.25)

    axes[0].set_ylabel("Pooled out-of-fold macro-F1")
    axes[-1].legend(frameon=False)
    fig.tight_layout()
    plt.show()
else:
    print("No completed setups to plot.")

## Cohort and class counts

Check this before interpreting performance. Small NEPC or AVPC classes can make estimates unstable even when nested CV is possible.

In [ ]:
counts_path = common.result_path("prediction_class_counts")
if os.path.exists(counts_path):
    class_counts = (
        pl.read_csv(counts_path)
        .filter(
            pl.col("target").is_in(TARGETS)
            & pl.col("space").is_in(SPACES)
            & pl.col("window").is_in(WINDOWS)
            & (pl.col("cohort_mode") == COHORT_MODE)
        )
        .sort(["target", "space", "class"])
    )
    display(class_counts)
else:
    print(f"No class-count table at {counts_path}")

## Selected hyperparameters and model artifacts

In [ ]:
rows = []
for setup in setups:
    target, space, window, model = setup
    meta_path = artifact_paths(*setup)[2]
    if not os.path.exists(meta_path):
        continue
    with open(meta_path) as handle:
        meta = json.load(handle)
    rows.append({
        "target": target,
        "space": space,
        "window": window,
        "model": model,
        "n_patients": meta["n_patients"],
        "n_features": meta["n_features"],
        "classes": ", ".join(meta["classes"]),
        "best_params": json.dumps(meta["final_best_params"], sort_keys=True),
        "model_path": meta["artifacts"]["model"],
    })

if rows:
    display(pl.DataFrame(rows).sort(["target", "model", "space"]))
else:
    print("No model metadata found for the requested setups.")